In [1]:
!pip3 install -q shap xgboost lightgbm geopandas scikit-learn matplotlib seaborn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import re

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
)
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("All imports successful")



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
All imports successful


In [3]:
listings = pd.read_csv("./content/listings.csv")
lsoa_geo = gpd.read_file("./content/LSOA.geojson")

try:
    lsoa_csv = pd.read_csv("./content/LSOA.csv")
    print(f"LSOA CSV : {lsoa_csv.shape}")
except Exception:
    lsoa_csv = None
    print("LSOA CSV : not found (continuing without it)")

print(f"Listings  : {listings.shape}")
print(f"LSOA GeoJSON : {lsoa_geo.shape}")

LSOA CSV : (5450, 11)
Listings  : (96182, 75)
LSOA GeoJSON : (5450, 10)


In [4]:
print("=" * 60)
print("LISTINGS — Column Overview")
print("=" * 60)
print(listings.dtypes.to_string())
print(f"\nPrice column sample:\n{listings['price'].head(10)}")
print(f"\nMissing values (top 20):\n{listings.isnull().sum().sort_values(ascending=False).head(20)}")

LISTINGS — Column Overview
id                                                int64
listing_url                                      object
scrape_id                                         int64
last_scraped                                     object
source                                           object
name                                             object
description                                      object
neighborhood_overview                            object
picture_url                                      object
host_id                                           int64
host_url                                         object
host_name                                        object
host_since                                       object
host_location                                    object
host_about                                       object
host_response_time                               object
host_response_rate                               object
host_acceptance_rate 

In [5]:
def clean_price(col):
    """Convert price strings like '$1,200.00' to float."""
    if col.dtype == "object":
        return (
            col.astype(str)
            .str.replace(r"[$,\u00a3\u20ac]", "", regex=True)
            .str.strip()
            .replace("", np.nan)
            .astype(float)
        )
    return col.astype(float)

listings["price"] = clean_price(listings["price"])


for c in ["cleaning_fee", "security_deposit", "extra_people"]:
    if c in listings.columns:
        listings[c] = clean_price(listings[c]).fillna(0)


listings = listings[listings["price"].notna() & (listings["price"] > 0)].copy()


q_low, q_high = listings["price"].quantile(0.01), listings["price"].quantile(0.99)
listings = listings[(listings["price"] >= q_low) & (listings["price"] <= q_high)].copy()

print(f"After cleaning: {listings.shape[0]} rows")
print(f"Price range: ${listings['price'].min():.0f} - ${listings['price'].max():.0f}")
print(f"Median price: ${listings['price'].median():.0f}")

After cleaning: 62136 rows
Price range: $30 - $1000
Median price: $130


In [6]:
CITY_CENTER = (51.5074, -0.1278)  


POIS = {
    "dist_city_center": CITY_CENTER,
    "dist_big_ben": (51.5007, -0.1246),
    "dist_tower_bridge": (51.5055, -0.0754),
    "dist_hyde_park": (51.5073, -0.1657),
    "dist_kings_cross": (51.5322, -0.1240),
    "dist_heathrow": (51.4700, -0.4543),
    "dist_canary_wharf": (51.5054, -0.0235),
}

def haversine(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


for name, (poi_lat, poi_lon) in POIS.items():
    listings[name] = haversine(
        listings["latitude"].values, listings["longitude"].values,
        poi_lat, poi_lon
    )


poi_cols = [c for c in POIS if c != "dist_city_center"]
listings["dist_nearest_poi"] = listings[poi_cols].min(axis=1)


from sklearn.neighbors import BallTree

coords = listings[["latitude", "longitude"]].values
tree = BallTree(np.radians(coords), metric="haversine")
listings["listing_density_1km"] = tree.query_radius(
    np.radians(coords), r=1.0 / 6371, count_only=True
) - 1  

print(f"Distance features added: {list(POIS.keys()) + ['dist_nearest_poi', 'listing_density_1km']}")

Distance features added: ['dist_city_center', 'dist_big_ben', 'dist_tower_bridge', 'dist_hyde_park', 'dist_kings_cross', 'dist_heathrow', 'dist_canary_wharf', 'dist_nearest_poi', 'listing_density_1km']


In [7]:
gdf_listings = gpd.GeoDataFrame(
    listings,
    geometry=gpd.points_from_xy(listings["longitude"], listings["latitude"]),
    crs="EPSG:4326",
)

lsoa_geo = lsoa_geo.to_crs("EPSG:4326")


gdf_joined = gpd.sjoin(gdf_listings, lsoa_geo, how="left", predicate="within")


if lsoa_csv is not None:
    geo_cols = lsoa_geo.columns.tolist()
    csv_cols = lsoa_csv.columns.tolist()
    print(f"LSOA GeoJSON columns: {geo_cols}")
    print(f"LSOA CSV columns:     {csv_cols}")

    key_candidates = ["LSOA11CD", "lsoa11cd", "LSOA_CODE", "lsoa_code", "code", "Code"]
    merge_key = None
    for k in key_candidates:
        if k in gdf_joined.columns and k in lsoa_csv.columns:
            merge_key = k
            break

    if merge_key:
        gdf_joined = gdf_joined.merge(lsoa_csv, on=merge_key, how="left", suffixes=("", "_lsoa"))
        print(f"Merged LSOA CSV on key: {merge_key}")
    else:
        print("Could not find a matching key — skipping LSOA CSV merge")

listings = pd.DataFrame(gdf_joined.drop(columns=["geometry"]))
print(f"After LSOA join: {listings.shape}")

LSOA GeoJSON columns: ['FID', 'LSOA21CD', 'LSOA21NM', 'LSOA21NMW', 'BNG_E', 'BNG_N', 'LAT', 'LONG', 'GlobalID', 'geometry']
LSOA CSV columns:     ['FID', 'LSOA21CD', 'LSOA21NM', 'LSOA21NMW', 'BNG_E', 'BNG_N', 'LAT', 'LONG', 'Shape__Area', 'Shape__Length', 'GlobalID']
Could not find a matching key — skipping LSOA CSV merge
After LSOA join: (62136, 94)


In [8]:

for col in ["host_response_rate", "host_acceptance_rate"]:
    if col in listings.columns:
        listings[col] = (
            listings[col].astype(str)
            .str.replace("%", "")
            .replace(["N/A", "nan", ""], np.nan)
            .astype(float)
        )


bool_map = {"t": 1, "f": 0, True: 1, False: 0}
bool_candidates = [
    "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
    "instant_bookable", "is_business_travel_ready", "has_availability",
    "require_guest_profile_picture", "require_guest_phone_verification",
]
for col in bool_candidates:
    if col in listings.columns:
        listings[col] = listings[col].map(bool_map).fillna(0).astype(int)


if "amenities" in listings.columns:
    listings["amenity_count"] = listings["amenities"].astype(str).apply(
        lambda x: len(re.findall(r'"([^"]+)"', x)) if x != "nan" else 0
    )


if "host_since" in listings.columns:
    listings["host_since"] = pd.to_datetime(listings["host_since"], errors="coerce")
    listings["host_days"] = (pd.Timestamp.now() - listings["host_since"]).dt.days


for col in ["description", "name", "neighborhood_overview"]:
    if col in listings.columns:
        listings[f"{col}_len"] = listings[col].astype(str).str.len()


review_cols = [c for c in listings.columns if c.startswith("review_scores")]
for c in review_cols:
    listings[c] = listings[c].fillna(listings[c].median())


listings["log_price"] = np.log1p(listings["price"])

print(f"Feature engineering complete — {listings.shape[1]} columns")

Feature engineering complete — 100 columns


In [9]:
NUM_FEATURES = [
    "accommodates", "bathrooms", "bedrooms", "beds", "guests_included",
    "minimum_nights", "maximum_nights", "availability_30", "availability_60",
    "availability_90", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm",
    "calculated_host_listings_count",
    "reviews_per_month",
    "host_response_rate", "host_acceptance_rate",
    "host_is_superhost", "host_identity_verified", "instant_bookable",
    "amenity_count", "host_days",
    "description_len", "name_len",
    "listing_density_1km",
    "latitude", "longitude",
]

DIST_FEATURES = [c for c in listings.columns if c.startswith("dist_")]
REVIEW_FEATURES = [c for c in listings.columns if c.startswith("review_scores")]


LSOA_NUM = [
    c for c in listings.columns
    if listings[c].dtype in ["float64", "int64"]
    and c not in NUM_FEATURES + DIST_FEATURES + REVIEW_FEATURES
    and c not in ["price", "log_price", "id", "scrape_id", "host_id", "index_right"]
    and "index" not in c.lower()
]


CAT_FEATURES = ["room_type", "neighbourhood_cleansed", "property_type"]
CAT_FEATURES = [c for c in CAT_FEATURES if c in listings.columns]

all_num = [c for c in NUM_FEATURES + DIST_FEATURES + REVIEW_FEATURES + LSOA_NUM
           if c in listings.columns]


label_encoders = {}
for col in CAT_FEATURES:
    le = LabelEncoder()
    listings[col + "_enc"] = le.fit_transform(listings[col].astype(str))
    label_encoders[col] = le

cat_enc = [c + "_enc" for c in CAT_FEATURES]
FEATURE_COLS = all_num + cat_enc


missing_pct = listings[FEATURE_COLS].isnull().mean()
drop_cols = missing_pct[missing_pct > 0.40].index.tolist()
FEATURE_COLS = [c for c in FEATURE_COLS if c not in drop_cols]
print(f"Dropped {len(drop_cols)} high-missing columns: {drop_cols}")


listings[FEATURE_COLS] = listings[FEATURE_COLS].apply(lambda c: c.fillna(c.median()))

X = listings[FEATURE_COLS].copy()
y = listings["log_price"].copy()

print(f"\nFinal feature matrix: X={X.shape}, y={y.shape}")
print(f"Features ({len(FEATURE_COLS)}):")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:3d}. {f}")

Dropped 3 high-missing columns: ['neighbourhood_group_cleansed', 'calendar_updated', 'license']

Final feature matrix: X=(62136, 64), y=(62136,)
Features (64):
    1. accommodates
    2. bathrooms
    3. bedrooms
    4. beds
    5. minimum_nights
    6. maximum_nights
    7. availability_30
    8. availability_60
    9. availability_90
   10. availability_365
   11. number_of_reviews
   12. number_of_reviews_ltm
   13. calculated_host_listings_count
   14. reviews_per_month
   15. host_response_rate
   16. host_acceptance_rate
   17. host_is_superhost
   18. host_identity_verified
   19. instant_bookable
   20. amenity_count
   21. host_days
   22. description_len
   23. name_len
   24. listing_density_1km
   25. latitude
   26. longitude
   27. dist_city_center
   28. dist_big_ben
   29. dist_tower_bridge
   30. dist_hyde_park
   31. dist_kings_cross
   32. dist_heathrow
   33. dist_canary_wharf
   34. dist_nearest_poi
   35. review_scores_rating
   36. review_scores_accuracy
   37. r

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

Train: 49708  |  Test: 12428


In [11]:
models = {
    "XGBoost": XGBRegressor(
        n_estimators=1000, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, early_stopping_rounds=50,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=1000, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbose=-1,
    ),
    "GBR": GradientBoostingRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, random_state=42,
    ),
}

results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")

    if name == "XGBoost":
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    elif name == "LightGBM":
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
    else:
        model.fit(X_train, y_train)

    y_pred_log = model.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    y_actual = np.expm1(y_test)

    r2 = r2_score(y_test, y_pred_log)
    mae = mean_absolute_error(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
    mape = mean_absolute_percentage_error(y_actual, y_pred) * 100

    results[name] = {"R2": r2, "MAE": mae, "RMSE": rmse, "MAPE": mape, "model": model}
    print(f"  R2    = {r2:.4f}")
    print(f"  MAE   = ${mae:.2f}")
    print(f"  RMSE  = ${rmse:.2f}")
    print(f"  MAPE  = {mape:.1f}%")


print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
summary = pd.DataFrame({k: {m: v for m, v in v.items() if m != "model"}
                         for k, v in results.items()}).T
print(summary.to_string())

best_name = max(results, key=lambda k: results[k]["R2"])
best_model = results[best_name]["model"]
print(f"\nBest model: {best_name} (R2 = {results[best_name]['R2']:.4f})")


Training XGBoost...
  R2    = 0.8310
  MAE   = $37.09
  RMSE  = $70.56
  MAPE  = 21.3%

Training LightGBM...
  R2    = 0.8243
  MAE   = $38.04
  RMSE  = $71.83
  MAPE  = 21.9%

Training GBR...
  R2    = 0.8183
  MAE   = $38.82
  RMSE  = $73.01
  MAPE  = 22.4%

MODEL COMPARISON
                R2        MAE       RMSE       MAPE
XGBoost   0.831049  37.089157  70.555842  21.273672
LightGBM  0.824318  38.039987  71.828126  21.910188
GBR       0.818307  38.821292  73.013885  22.388337

Best model: XGBoost (R2 = 0.8310)


In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import clone

print(f"\nRunning 5-Fold Cross-Validation for {best_name} (with Shuffling)...")


kf = KFold(n_splits=5, shuffle=True, random_state=42)


cv_model = clone(best_model)



if hasattr(cv_model, "early_stopping_rounds"):
    cv_model.set_params(early_stopping_rounds=None)


cv_scores = cross_val_score(
    cv_model, 
    X, 
    y, 
    cv=kf, 
    scoring="r2", 
    n_jobs=-1
)


print("-" * 30)
print(f"CV R2 scores : {cv_scores.round(4)}")
print(f"Mean R2      : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print("-" * 30)

if cv_scores.std() > 0.02:
    print("⚠️ Warning: High variance between folds. Check for outliers or geographic biases.")
else:
    print("✅ Model is stable across all data segments.")


In [ ]:
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_actual = np.expm1(y_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(y_actual, y_pred, alpha=0.2, s=5, color="#2563eb")
lims = [0, min(y_actual.quantile(0.99), y_pred.max())]
ax.plot(lims, lims, "r--", lw=2)
ax.set_xlabel("Actual Price ($)")
ax.set_ylabel("Predicted Price ($)")
ax.set_title(f"Actual vs Predicted — {best_name}\nR2 = {results[best_name]['R2']:.4f}")
ax.set_xlim(lims); ax.set_ylim(lims)

residuals = y_actual - y_pred
ax = axes[1]
ax.hist(residuals, bins=80, color="#2563eb", edgecolor="white", alpha=0.8)
ax.axvline(0, color="red", linestyle="--", lw=2)
ax.set_xlabel("Residual ($)"); ax.set_title("Residual Distribution")

ax = axes[2]
ax.scatter(y_pred, residuals, alpha=0.2, s=5, color="#2563eb")
ax.axhline(0, color="red", linestyle="--", lw=2)
ax.set_xlabel("Predicted Price ($)"); ax.set_ylabel("Residual ($)")
ax.set_title("Residuals vs Predicted")

plt.tight_layout()
plt.savefig("./content/prediction_plots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
    imp = imp.sort_values(ascending=True).tail(25)

    fig, ax = plt.subplots(figsize=(8, 8))
    imp.plot(kind="barh", color="#2563eb", ax=ax)
    ax.set_title(f"Top 25 Feature Importances — {best_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("./content/feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
print("Computing SHAP values...")

shap_sample_size = min(2000, X_test.shape[0])
X_shap = X_test.sample(shap_sample_size, random_state=42)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_shap)


fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, show=False, max_display=20)
plt.title(f"SHAP Summary — {best_name}")
plt.tight_layout()
plt.savefig("./content/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()


fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False, max_display=20)
plt.title(f"SHAP Feature Importance (Mean |SHAP|) — {best_name}")
plt.tight_layout()
plt.savefig("./content/shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
mean_shap = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_shap)[::-1][:6]
top_features = [FEATURE_COLS[i] for i in top_idx]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for feat, ax in zip(top_features, axes.flatten()):
    shap.dependence_plot(feat, shap_values, X_shap, ax=ax, show=False)
    ax.set_title(f"SHAP Dependence: {feat}")

plt.suptitle(f"SHAP Dependence Plots — Top 6 Features ({best_name})", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("./content/shap_dependence.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
idx = 0
single_pred = np.expm1(best_model.predict(X_shap.iloc[[idx]]))[0]
actual_val = np.expm1(y_test.loc[X_shap.index[idx]])

print(f"Sample listing #{X_shap.index[idx]}")
print(f"  Predicted : ${single_pred:.2f}")
print(f"  Actual    : ${actual_val:.2f}")

shap.initjs()
explanation = shap.Explanation(
    values=shap_values[idx],
    base_values=(explainer.expected_value
                 if np.isscalar(explainer.expected_value)
                 else explainer.expected_value[0]),
    data=X_shap.iloc[idx].values,
    feature_names=FEATURE_COLS,
)

fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.waterfall(explanation, max_display=15, show=False)
plt.title(f"SHAP Waterfall — Listing #{X_shap.index[idx]} (Pred: ${single_pred:.0f})")
plt.tight_layout()
plt.savefig("./content/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n" + "=" * 60)
print("DISTANCE FEATURE ANALYSIS")
print("=" * 60)

dist_cols_in_model = [c for c in FEATURE_COLS
                      if c.startswith("dist_") or c == "listing_density_1km"]
dist_shap = pd.DataFrame(shap_values, columns=FEATURE_COLS)[dist_cols_in_model]

print("\nMean |SHAP| for distance features:")
dist_importance = dist_shap.abs().mean().sort_values(ascending=False)
for feat, val in dist_importance.items():
    print(f"  {feat:30s} : {val:.4f}")

print("\nCorrelation with price:")
for col in dist_cols_in_model:
    corr = listings[col].corr(listings["price"])
    print(f"  {col:30s} : {corr:+.4f}")

In [ ]:
print("\n" + "=" * 60)
print("FINAL MODEL SUMMARY")
print("=" * 60)
print(f"  Best Model       : {best_name}")
print(f"  Test R2          : {results[best_name]['R2']:.4f}")
print(f"  Test MAE         : ${results[best_name]['MAE']:.2f}")
print(f"  Test RMSE        : ${results[best_name]['RMSE']:.2f}")
print(f"  Test MAPE        : {results[best_name]['MAPE']:.1f}%")
print(f"  CV R2 (5-fold)   : {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"  Features Used    : {len(FEATURE_COLS)}")
print(f"  Training Samples : {X_train.shape[0]}")
print(f"  Test Samples     : {X_test.shape[0]}")
print(f"  Distance Features: {len(dist_cols_in_model)}")
print("=" * 60)
print("All plots saved to /content/")